# Start Nomination Simulation (Base CSV Workflow)

This notebook runs embedding-based start nomination simulation from existing Plan B CSVs.

The script now writes one primary analysis table: `start_nomination_base.csv`.
Each row is a subset-level paired comparison for `(selector, uncertainty_policy)`.

- `matched_*` = start-conditioned uncertainty outcome (for the nominated start image)
- `random_avg_*` = matched random baseline subset mean
- `delta_*` = matched uncertainty minus random baseline (iterations improvement is sign-normalized)
- `*_vs_uncertainty_mean` columns are secondary diagnostics only


In [ ]:
from pathlib import Path
import json
import shlex
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments.analysis.hierarchical_ci import hierarchical_bootstrap_grouped_deltas


def resolve_repo_root(start: Path | None = None) -> Path:
    probe = (start or Path.cwd()).resolve()
    for cand in [probe, *probe.parents]:
        if (cand / 'experiments' / 'scripts').exists() and (cand / 'experiments' / 'analysis').exists():
            return cand
    raise FileNotFoundError(
        f"Could not resolve repo root from cwd={Path.cwd()}. "
        "Expected to find experiments/scripts and experiments/analysis in a parent directory."
    )


REPO_ROOT = resolve_repo_root()
FIG_DIR = REPO_ROOT / 'figures' / 'start_nomination_simulation'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Resolved REPO_ROOT:', REPO_ROOT)

ENV_PYTHON = '/data/ddmg/users/gtorpey/envs/mvseg-ordering-env/bin/python'


In [ ]:
# ---- Config ----
procedure = 'random_vs_uncertainty_final'
ablation = 'pretrained_baseline'
uncertainty_policies = ['curriculum']

# Random source defaults to same procedure/ablation unless overridden.
random_procedure = None
random_ablation = None
random_policy = 'random'

dataset = 'ACDC'            # or None for all
mega_slicing = None         # 'midslice' / 'maxslice' / None

encoder_config_path = 'experiments/encoder_configs/vit_default.yaml'
encoder_config_key = None   # optional top-level key
data_split = 'train'
device = 'cuda'

selectors = ['closest_centroid', 'farthest_centroid', 'medoid', 'max_mean_distance']
seed = 0
n_boot = 2000
alpha = 0.05

out_dir = REPO_ROOT / 'figures' / 'start_nomination_simulation'
base_csv_name = 'start_nomination_base.csv'


In [ ]:
cmd = [
    ENV_PYTHON,
    '-m', 'experiments.analysis.start_nomination_simulation',
    '--procedure', procedure,
    '--ablation', ablation,
    '--random-policy', random_policy,
    '--encoder-config-path', encoder_config_path,
    '--data-split', data_split,
    '--device', device,
    '--seed', str(seed),
    '--out-dir', str(out_dir),
    '--base-csv-name', base_csv_name,
]

cmd.extend(['--policy', *uncertainty_policies])
cmd.extend(['--selectors', *selectors])
if random_procedure is not None:
    cmd.extend(['--random-procedure', random_procedure])
if random_ablation is not None:
    cmd.extend(['--random-ablation', random_ablation])
if dataset is not None:
    cmd.extend(['--dataset', dataset])
if mega_slicing is not None:
    cmd.extend(['--mega-slicing', mega_slicing])
if encoder_config_key is not None:
    cmd.extend(['--encoder-config-key', encoder_config_key])

print('Running command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


In [ ]:
base_path = out_dir / base_csv_name
meta_path = out_dir / 'run_metadata.json'

base = pd.read_csv(base_path)
metadata = json.loads(meta_path.read_text()) if meta_path.exists() else {}

print('base:', base.shape)
print('metadata keys:', sorted(metadata.keys()))

required_cols = [
    'family', 'task_id', 'subset_index', 'selector', 'uncertainty_policy',
    'matched_vs_random', 'coverage_status',
    'matched_final_dice', 'matched_iterations_used', 'matched_goal_rate', 'matched_goal_auc',
    'random_avg_final_dice', 'random_avg_iterations_used', 'random_avg_goal_rate', 'random_avg_goal_auc',
    'delta_final_dice', 'delta_iterations_used_improvement', 'delta_goal_rate', 'delta_goal_auc',
]
missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f'Missing expected base CSV columns: {missing}')

base.head(5)


In [ ]:
# Headline summary with hierarchical bootstrap deltas (primary vs random)
keys = ['selector', 'uncertainty_policy']

coverage = (
    base.groupby(keys, as_index=False)
    .agg(
        n_total=('matched_vs_random', 'size'),
        n_matched=('matched_vs_random', 'sum'),
        coverage_rate=('matched_vs_random', 'mean'),
    )
)

metric_to_delta = {
    'final_dice': 'delta_final_dice',
    'iterations_used_improvement': 'delta_iterations_used_improvement',
    'goal_rate': 'delta_goal_rate',
    'goal_auc': 'delta_goal_auc',
}

primary = base[base['matched_vs_random'] == True].copy()
primary_long = primary.melt(
    id_vars=['family', 'task_id', 'subset_index', 'selector', 'uncertainty_policy'],
    value_vars=list(metric_to_delta.values()),
    var_name='delta_col',
    value_name='delta_value',
)
col_to_metric = {v: k for k, v in metric_to_delta.items()}
primary_long['metric'] = primary_long['delta_col'].map(col_to_metric)
primary_long = primary_long.dropna(subset=['metric', 'delta_value']).copy()

primary_boot = hierarchical_bootstrap_grouped_deltas(
    primary_long,
    value_col='delta_value',
    group_cols=['selector', 'uncertainty_policy', 'metric'],
    dataset_col='family',
    task_col='task_id',
    n_boot=n_boot,
    seed=seed,
    alpha=alpha,
)

dataset_delta_summary = primary_boot['dataset_summary'].copy()
global_delta_summary = primary_boot['global_summary'].copy()

headline = coverage.merge(
    global_delta_summary[global_delta_summary['metric'].isin(['final_dice', 'iterations_used_improvement'])]
    .pivot_table(
        index=['selector', 'uncertainty_policy'],
        columns='metric',
        values='mean',
        aggfunc='first',
    )
    .rename(columns={
        'final_dice': 'delta_final_dice_mean_hier',
        'iterations_used_improvement': 'delta_iterations_improvement_mean_hier',
    })
    .reset_index(),
    on=['selector', 'uncertainty_policy'],
    how='left',
).sort_values(['selector', 'uncertainty_policy']).reset_index(drop=True)

print('Global hierarchical summary (all metrics):')
display(global_delta_summary.sort_values(['metric', 'selector', 'uncertainty_policy']).reset_index(drop=True))
print('Headline table:')
headline


In [ ]:
# Coverage diagnostics by status
coverage_detail = (
    base.groupby(['selector', 'uncertainty_policy', 'coverage_status'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

totals = (
    coverage_detail.groupby(['selector', 'uncertainty_policy'], as_index=False)['count']
    .sum()
    .rename(columns={'count': 'total'})
)
coverage_detail = coverage_detail.merge(totals, on=['selector', 'uncertainty_policy'], how='left')
coverage_detail['rate'] = coverage_detail['count'] / coverage_detail['total'].replace({0: np.nan})
coverage_detail.sort_values(['selector', 'uncertainty_policy', 'coverage_status']).reset_index(drop=True)


In [ ]:
# Dataset-level hierarchical bootstrap delta bars (primary vs random)
plot_base = dataset_delta_summary.copy()
if plot_base.empty:
    print('No matched rows to plot.')
else:
    metric_specs = [
        ('final_dice', 'Delta Final Dice (hierarchical mean ± CI)'),
        ('iterations_used_improvement', 'Delta Iterations Improvement (hierarchical mean ± CI)'),
    ]
    for metric_name, ylabel in metric_specs:
        plot_df = (
            plot_base[plot_base['metric'] == metric_name]
            .sort_values(['selector', 'uncertainty_policy', 'family'])
            .reset_index(drop=True)
        )
        if plot_df.empty:
            continue

        labels = [f"{r.selector}\n{r.uncertainty_policy}\n{r.family}" for r in plot_df.itertuples(index=False)]
        means = plot_df['mean'].to_numpy(dtype=float)
        lo = plot_df['ci_lo'].to_numpy(dtype=float)
        hi = plot_df['ci_hi'].to_numpy(dtype=float)
        yerr = [means - lo, hi - means]

        fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(labels)), 4.5))
        ax.bar(range(len(labels)), means, yerr=yerr, capsize=4)
        ax.axhline(0.0, color='black', linewidth=0.8)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(f'Start nomination simulation (hierarchical): {metric_name}')
        fig.tight_layout()

        out_path = FIG_DIR / f'dataset_hier_{metric_name}.png'
        fig.savefig(out_path, dpi=200)
        plt.show()
        plt.close(fig)
        print('Saved', out_path)


In [ ]:
# Matched uncertainty vs random-average scatter plots
plot_base = base[base['matched_vs_random'] == True].copy()
if plot_base.empty:
    print('No matched rows to plot.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    x = plot_base['random_avg_final_dice'].to_numpy(dtype=float)
    y = plot_base['matched_final_dice'].to_numpy(dtype=float)
    axes[0].scatter(x, y, alpha=0.7)
    mn, mx = np.nanmin(np.r_[x, y]), np.nanmax(np.r_[x, y])
    axes[0].plot([mn, mx], [mn, mx], linestyle='--', linewidth=1)
    axes[0].set_xlabel('Random avg final dice')
    axes[0].set_ylabel('Matched (start-conditioned) final dice')
    axes[0].set_title('Final Dice')

    x = plot_base['random_avg_iterations_used'].to_numpy(dtype=float)
    y = plot_base['matched_iterations_used'].to_numpy(dtype=float)
    axes[1].scatter(x, y, alpha=0.7)
    mn, mx = np.nanmin(np.r_[x, y]), np.nanmax(np.r_[x, y])
    axes[1].plot([mn, mx], [mn, mx], linestyle='--', linewidth=1)
    axes[1].set_xlabel('Random avg iterations used')
    axes[1].set_ylabel('Matched (start-conditioned) iterations used')
    axes[1].set_title('Iterations Used')

    fig.tight_layout()
    out_path = FIG_DIR / 'matched_vs_random_scatter.png'
    fig.savefig(out_path, dpi=200)
    plt.show()
    plt.close(fig)
    print('Saved', out_path)


In [ ]:
# Optional secondary diagnostics: matched uncertainty vs uncertainty-policy subset mean (hierarchical)
secondary_metric_to_delta = {
    'final_dice': 'delta_final_dice_vs_uncertainty_mean',
    'iterations_used_improvement': 'delta_iterations_used_improvement_vs_uncertainty_mean',
    'goal_rate': 'delta_goal_rate_vs_uncertainty_mean',
    'goal_auc': 'delta_goal_auc_vs_uncertainty_mean',
}

secondary_cols = ['matched_vs_uncertainty_mean', *secondary_metric_to_delta.values()]
if all(col in base.columns for col in secondary_cols):
    sec = base[base['matched_vs_uncertainty_mean'] == True].copy()
    if sec.empty:
        print('No matched secondary rows available.')
    else:
        sec_long = sec.melt(
            id_vars=['family', 'task_id', 'subset_index', 'selector', 'uncertainty_policy'],
            value_vars=list(secondary_metric_to_delta.values()),
            var_name='delta_col',
            value_name='delta_value',
        )
        sec_col_to_metric = {v: k for k, v in secondary_metric_to_delta.items()}
        sec_long['metric'] = sec_long['delta_col'].map(sec_col_to_metric)
        sec_long = sec_long.dropna(subset=['metric', 'delta_value']).copy()

        sec_boot = hierarchical_bootstrap_grouped_deltas(
            sec_long,
            value_col='delta_value',
            group_cols=['selector', 'uncertainty_policy', 'metric'],
            dataset_col='family',
            task_col='task_id',
            n_boot=n_boot,
            seed=seed,
            alpha=alpha,
        )
        sec_global = sec_boot['global_summary'].copy()
        sec_dataset = sec_boot['dataset_summary'].copy()

        print('Secondary global hierarchical summary:')
        display(sec_global.sort_values(['metric', 'selector', 'uncertainty_policy']).reset_index(drop=True))
        print('Secondary dataset hierarchical summary:')
        sec_dataset.sort_values(['metric', 'selector', 'uncertainty_policy', 'family']).reset_index(drop=True)
else:
    print('Secondary uncertainty-mean columns are not present in this base CSV.')
